# Spice Hoes — ComfyUI GPU worker\nRun on a Colab GPU. Installs ComfyUI + ComfyUI-GGUF and opens a temporary Cloudflare tunnel. No hosted generation API key is required. Colab policies and model licenses still apply.

In [ ]:
!nvidia-smi\n!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI || true\n%cd /content/ComfyUI\n!pip -q install -r requirements.txt\n!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF || true\n!pip -q install -r custom_nodes/ComfyUI-GGUF/requirements.txt || true\n

## Models\nPut your selected Wan/LTX GGUF or quantized checkpoint in the model directory expected by the workflow. Prefer a Wan low-VRAM workflow on free-tier GPUs. Mount Drive only if you want persistent model storage.

In [ ]:
# Optional persistent storage\n# from google.colab import drive\n# drive.mount('/content/drive')\n

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared\n!chmod +x /usr/local/bin/cloudflared\nimport subprocess, time, re\ncomfy = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'], cwd='/content/ComfyUI')\ntime.sleep(8)\ntunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\nurl = None\ndeadline = time.time() + 60\nwhile time.time() < deadline and not url:\n    line = tunnel.stdout.readline()\n    print(line, end='')\n    m = re.search(r'https://[-a-z0-9]+\\.trycloudflare\\.com', line)\n    if m: url = m.group(0)\nprint('\\nCOMFYUI_BASE_URL=', url)\n

Copy the printed `COMFYUI_BASE_URL` into the Spice runtime. Load/test your Wan or LTX workflow at the same URL, export **API Format**, and set `COMFYUI_WORKFLOW` to that JSON.